# Eval-Driven Development

Unit tests check code paths. They cannot tell you whether a *model* still
classifies sentiment correctly after you tweaked a prompt or swapped a model.
For that you need a golden set and an accuracy gate. This notebook runs one
and shows how it blocks a regression in CI.

## Why not a unit test?

A unit test asserts an exact output. LLM outputs vary and are judged by
*accuracy over a set*, not exact equality on one input. The eval harness scores
a `predict_fn` against labelled examples and records accuracy + p50 latency.

In [ ]:
import os, sys
sys.path.insert(0, "../src")

from genai_prod_kit.evals.runner import load_golden, run_eval, append_run

golden = load_golden("../src/genai_prod_kit/evals/golden/toy_sentiment.jsonl")
print(f"{len(golden)} labelled examples")
print(golden[0])

## Run the eval against the real model

`run_eval` takes any `predict_fn(text) -> str`. Here it wraps a Gemini call
through the Gateway, so the eval run is itself observable.

In [ ]:
from genai_prod_kit.gateway import call_llm
from genai_prod_kit.providers.gemini import GeminiProvider
from genai_prod_kit.sinks.jsonl import JsonlSink
from genai_prod_kit.prompts.registry import get_prompt

provider = GeminiProvider(os.environ["GEMINI_API_KEY"])
sink = JsonlSink("nb03_invocations.jsonl")
version, tmpl = get_prompt("toy_sentiment")

def classify(text: str) -> str:
    res = call_llm(tmpl.format(text=text), provider=provider, sink=sink,
                   feature="toy_sentiment", model="gemini-2.5-flash",
                   prompt_version=version)
    return res.text.strip().lower()

run = run_eval(golden, classify, feature="toy_sentiment", note="nb03")
print(f"accuracy   = {run.accuracy:.1%}  ({run.golden_count} items)")
print(f"p50 latency= {run.latency_p50_ms:.0f} ms")
print(f"git_sha    = {run.git_sha[:8]}")

## Persist the run — the gate compares the last two

Each run is appended to `eval_runs.jsonl`. The regression gate (`evals/gate.py`)
reads the last two runs and exits non-zero if accuracy dropped beyond the
threshold. Wired into CI, that turns a silent quality regression into a failed
check that blocks the merge.

In [ ]:
append_run(run, "nb03_eval_runs.jsonl")
print("appended to nb03_eval_runs.jsonl")

# Simulate the gate logic on two runs (no real CI needed here).
from genai_prod_kit.evals import gate
import json

# write a baseline + a regressed run to a scratch file to show a FAIL
with open("nb03_gate_demo.jsonl", "w", encoding="utf-8") as f:
    f.write(json.dumps({"accuracy": 1.00, "git_sha": "baseline0"}) + "\n")
    f.write(json.dumps({"accuracy": 0.80, "git_sha": "regressed"}) + "\n")

exit_code = gate.main("nb03_gate_demo.jsonl")
print("gate exit code:", exit_code, "(non-zero blocks the merge)")

## Takeaway

The golden set is the contract for model behaviour; the gate enforces it on
every change. You change a prompt or a model, the eval runs, and a regression
can no longer reach production unnoticed.